# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL for the FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access key metadata fields
print("Dataset Title: ", dataset.metadata.name)
print("Description: ", dataset.metadata.description)
print("Published: ", dataset.metadata.datePublished)
print("Version: ", dataset.metadata.version)
print("Keywords: ", getattr(dataset.metadata, 'keywords', None) or [])

# Show citation
print("\nCite as:", getattr(dataset.metadata, 'citeAs', ''))

## 2. Data Overview
Review available record sets, fields, and their IDs. All dataset structures are referenced by their `@id`.

In [ ]:
# List record sets available in the dataset, along with their fields and columns.

print("All available Record Sets and Field/Column IDs:\n")
record_sets = list(dataset.metadata.recordSet)
if not record_sets:
    raise RuntimeError("No record sets found in the dataset schema.")

for idx, rs in enumerate(record_sets):
    print(f"[{idx}] Record Set @id: {rs['@id']}")
    print(f"    Name: {rs.get('name', '')}")
    # Fields
    if 'field' in rs and rs['field']:
        print("    Fields:")
        for field in rs['field']:
            fid = field['@id']
            print(f"      - Field @id: {fid} (name: {field.get('name','')}, dataType: {field.get('dataType','')})")
    # Columns (if present)
    if 'column' in rs and rs['column']:
        print("    Columns:")
        for col in rs['column']:
            cid = col['@id']
            print(f"      - Column @id: {cid} (name: {col.get('name','')})")
    print()

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame for analysis. All record set, field, and column entities are referenced by their `@id`.

In [ ]:
# Collect all record set @ids for extraction
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"\nLoaded Record Set: {record_set_id}")
            print(f"Columns: {df.columns.tolist()}")
            print(df.head(3))
        else:
            print(f"\nNo records found for Record Set {record_set_id}.")
    except Exception as e:
        print(f"\nRecord Set {record_set_id} -- FAILED to load: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply data processing such as filtering, normalization, and grouping using field and column `@id`s. Select a record set, identify a numeric field (e.g., 'MW' for molecular weight), and a grouping field for further analysis.

In [ ]:
# Choose a record set and fields by @id
# (Update these IDs based on the printed overview above for your dataset)

# Example IDs (change as appropriate for your dataset):
example_record_set_id = record_set_ids[0]  # Adjust index as per actual overview
df = dataframes[example_record_set_id]

# Try to infer numeric and grouping fields from the DataFrame
numeric_field = None
group_field = None
for col in df.columns:
    if 'mw' in col.lower() or 'weight' in col.lower() or col.lower().startswith('cr:mw'):
        numeric_field = col
    elif 'protein' in col.lower() or 'desc' in col.lower() or 'group' in col.lower():
        group_field = col

if numeric_field is None:
    # Fallback: Just pick the first float/integer-typed column
    for col in df.select_dtypes(include=['number']).columns:
        numeric_field = col
        break

assert numeric_field is not None, "No obvious numeric field found in selected Record Set."

print(f"Selected Numeric Field for Analysis: '{numeric_field}'")
if group_field:
    print(f"Selected Grouping Field: '{group_field}'")

# Filter records: MW > threshold
threshold = df[numeric_field].mean() if df[numeric_field].mean() == df[numeric_field].mean() else 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head(3))

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head(3))

# Optionally, group by some field
if group_field is not None and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by '{group_field}':")
    print(grouped_df.head(3))

## 5. Visualization
Visualize the distribution of the chosen numeric field and/or grouped statistics.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field], bins=40, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Optionally, mean value by group
if group_field and group_field in df.columns:
    plt.figure(figsize=(10,5))
    grp = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
    sns.barplot(y=grp.index, x=grp.values)
    plt.title(f'Top 10 {group_field} by Mean {numeric_field}')
    plt.xlabel(f'Mean {numeric_field}')
    plt.ylabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to access the FAIR<sup>2</sup> protein dataset, examine its structure using `@id` references, and perform initial exploratory analysis. You can continue by applying domain-specific analyses, building ML features, or integrating into downstream pipelines.